In [ ]:
!pip install -U pip setuptools wheel
!pip install -U moshi huggingface_hub

# Run this to fix bfloat16 issue

In [ ]:
fixed_code = '''# Copyright (c) Kyutai, all rights reserved.
# This source code is licensed under the license found in the
# LICENSE file in the root directory of this source tree.

import torch
from torch import nn


class QLinear(nn.Module):
    def __init__(self, linear: nn.Linear):
        super().__init__()
        from bitsandbytes import functional as bnbF
        weight = linear.weight
        assert weight.data.dtype.is_floating_point
        assert linear.bias is None
        CB, SCB, _ = bnbF.int8_vectorwise_quant(weight.data.to(torch.float16))
        self.weight = nn.Parameter(CB, requires_grad=False)
        self.weight_scb = nn.Parameter(SCB, requires_grad=False)

    def forward(self, x):
        import bitsandbytes as bnb
        state = bnb.MatmulLtState()
        state.CB = self.weight
        assert isinstance(state.CB, torch.Tensor)

        # Fix for Tesla T4: weight_scb gets corrupted to bfloat16, force back to float32
        if self.weight_scb.dtype != torch.float:
            self.weight_scb.data = self.weight_scb.data.float()

        state.SCB = self.weight_scb
        assert isinstance(state.SCB, torch.Tensor)
        state.has_fp16_weights = False
        y = bnb.matmul(x.half(), state.CB, state=state)
        assert isinstance(y, torch.Tensor)
        return y


def replace_linear_with_qlinear(module):
    """Recursively replace all Linear layers with QLinear layers."""
    for name, child in module.named_children():
        if isinstance(child, nn.Linear):
            setattr(module, name, QLinear(child))
        elif isinstance(child, QLinear):
            child.float()
        else:
            replace_linear_with_qlinear(child)
'''

with open('/usr/local/lib/python3.12/dist-packages/moshi/utils/quantize.py', 'w') as f:
    f.write(fixed_code)

print("File patched successfully!")

In [ ]:
import os
from huggingface_hub import login

token = "your_hf_token"
if token:
    login(token)
    print("HF login successful")
else:
    print("No HF token found, trying anonymous access")

# As kaggle block localhost connection. So the run the model as subprocess then create a public url to access moshi

In [ ]:
import subprocess
import time

# Start server in background
proc = subprocess.Popen(
    ["python", "-m", "moshi.server", "--hf-repo", "kyutai/moshiko-pytorch-q8"],
    stdout=open("moshi.log", "w"),
    stderr=subprocess.STDOUT
)

# Wait for it to load (model takes ~60 seconds)
print("Waiting for server to start...")
time.sleep(60)

# Check logs
!cat moshi.log

In [ ]:
# Get free token from https://dashboard.ngrok.com/signup
!ngrok authtoken your_auth_token_from_ngrok

from pyngrok import ngrok
public_url = ngrok.connect(8998)
print(f"✓ Access Moshi here: {public_url}")